# 04: Row-Level Security with Row Filters

**Exam objective:** Understand column-level masking and row-level security 
to restrict data visibility based on user groups.

**Scope of this notebook:** Row filters. Column masking is covered in 03.

**Free Edition note:** Same limitation as 03 — filter behavior is observed 
from the owner's perspective; non-owner behavior is reasoned about.

In [0]:
USE CATALOG certprep;
USE SCHEMA governance;

-- Recreate the table fresh for this exercise
DROP TABLE IF EXISTS sales_data;

CREATE TABLE sales_data (
  sale_id INT,
  region STRING,
  amount DECIMAL(10,2),
  customer_email STRING,
  sale_date DATE
);

INSERT INTO sales_data VALUES
  (1, 'North', 1250.00, 'alice@example.com', '2026-01-15'),
  (2, 'South', 890.50,  'bob@example.com',   '2026-01-16'),
  (3, 'East',  2100.75, 'carol@example.com', '2026-01-17'),
  (4, 'West',  1575.25, 'dan@example.com',   '2026-01-18'),
  (5, 'North', 3200.00, 'eve@example.com',   '2026-01-19'),
  (6, 'South', 450.25,  'frank@example.com', '2026-01-20');

SELECT * FROM sales_data;

In [0]:
-- Created new group: north_region_team
SHOW GROUPS;

In [0]:
-- Row filter: analysts see everything, north_region_team sees only North, anyone else sees nothing
CREATE OR REPLACE FUNCTION region_filter(region STRING)
RETURN
    CASE
        WHEN is_account_group_member('analysts') THEN TRUE
        WHEN is_account_group_member('north_region_team') THEN region = 'North'
        ELSE FALSE
    END;

**Prediction question**: if you are the owner (and not in either group), what should `SELECT * FROM sales_data` return after the filter is attached? Why? Then separately: if you were in `north_region_team` only, what should you see?

**Answers**:
    - The owner would see now rows if they are not a member of any group included in the filter. The function checks for group membership status, in which case the owner's lack of membership yields a false result.
    - A member of the `north_region_team` would only see rows where the value in the `region` column is equal to 'North'

In [0]:
-- Test the UDF directly first
SELECT 
  region,
  region_filter(region) AS would_be_visible
FROM sales_data;

In [0]:
ALTER TABLE sales_data
SET ROW FILTER region_filter ON (region);

In [0]:
-- Select with row filter on
SELECT * FROM sales_data;

As I answered, I do not see any rows despite being the owner of the table. The row filter excludes me from seeing rows as I am a member of neither the `analysts` group nor the `north_region_team` group.

In [0]:
-- Inspect the table's metadata
DESCRIBE TABLE EXTENDED sales_data;

Table metadata will display the row filter. Looks like this:
```
|_col_name___|_data_type_________________________________|
|_Row Filter_|_`catalog`.`schema`.`function` ON (column)_|
```

In [0]:
-- Recreate the email mask from exercise 03
CREATE OR REPLACE FUNCTION mask_email(email STRING)
RETURN
  CASE
    WHEN is_account_group_member('analysts') THEN email
    ELSE CONCAT('***@', SPLIT(email, '@')[1])
  END;

ALTER TABLE sales_data
ALTER COLUMN customer_email
SET MASK mask_email;

In [0]:
SELECT * FROM certprep.governance.sales_data;

Having added myself now to the `north_region_team` group, I can see the North rows with the masked customer_email values

In [0]:
-- Modify the filter: change to allow east region for north_region_team as well
CREATE OR REPLACE FUNCTION region_filter(region STRING)
RETURN
  CASE
    WHEN is_account_group_member('analysts') THEN TRUE
    WHEN is_account_group_member('north_region_team') THEN region IN ('North', 'East')
    ELSE FALSE
  END;

In [0]:
-- The filter is attached to the table by reference so no need to re-attach.
SELECT * FROM sales_data;

With the update made to the filter, I can now see East rows as a member of `north_region_team`.

In [0]:
-- Remove the filter
ALTER TABLE sales_data DROP ROW FILTER;

In [0]:
-- All rows visible again (column mask still active)
SELECT * FROM sales_data;

In [0]:
-- 1. Detach the column mask
ALTER TABLE sales_data ALTER COLUMN customer_email DROP MASK;

-- 2. Now drop the functions
DROP FUNCTION IF EXISTS region_filter;
DROP FUNCTION IF EXISTS mask_email;

-- 3. Verify
SHOW FUNCTIONS IN certprep.governance;

## Self Check Questions
1. A row filter UDF takes columns from the table as arguments. What's the implication if a row filter references columns that aren't passed in the ON (...) clause? Does it work?
2. You attach a row filter that returns FALSE for all rows. A user runs SELECT COUNT(*) FROM sales_data. What does it return? Does the count include filtered-out rows?
3. Row filters and column masks can be combined on the same table. Conceptually, which executes first, and why does that order matter for performance?
4. A user who is the owner of the table runs a query. Are they subject to the row filter? How does this differ from the owner's relationship with table-level privileges (which they have implicitly)?
5. You modify the underlying UDF of an attached row filter (e.g., change the CASE logic). Do you need to re-attach the filter to the table for the change to take effect? Why or why not?

## Self Check Answers
1. No. A row filter UDF can only reference columns that are passed to it via the `ON (...)` clause. Referencing a column that is not passed through the clause results in an error when trying to attach the filter. UC validates this at attach time to see which columns the filter depends on. This way, UC can use that metadata for understanding of the filter's data-access pattern and for query optimization. 
2. Returns zero. RLS is still in place, thus the query only touches rows that the filter returns TRUE for. In this case, the filter returns FALSE for all rows, and so no rows may be counted.
3. Row filters execute first. This allows for column masking to be run against a subset of the table's rows as opposed to all rows. It would be a waste of resources and nonsensical to mask a column in every row only to return a fraction of the total rows to a filtered user. 
4. Table owners are subject to row filters. A row filter checks for group membership, not table-level privileges, and determines which rows a user can see after they've been granted access to the table. Table-level privileges deal with whether groups/users may access a table at all, which table owners can do implicitly. The table owner must be a member of a group the function returns either TRUE or a partial filter for, otherwise they will fall into the FALSE category and see no rows. 
5. Modified UDFs do not need to be reattached. The table they are attached to references them, like a function call in a Python file. Modification changes the function's behavior, not the table's reference to the function. 